# Fine-tune DistilBERT for Transaction Categorization

**Goal:** Train a DistilBERT model to categorize bank transaction descriptions into 10 categories.

**Training data:** 68k realistic US bank-format transactions (DoDataThings dataset on HuggingFace)

**Runtime:** ~15 minutes on Colab T4 GPU

**Output:** A model you download and use as a fallback alongside the sklearn pipeline.

## How to use
1. `Runtime → Run all`
2. At the end, download the model zip file
3. Extract it into `apps/ml-service/data/distilbert-model/`

In [ ]:
# Install deps — don't force-upgrade torch/pandas/numpy (Colab ships them)
!pip install -q "transformers" "accelerate" "evaluate" "tqdm" "scikit-learn" "sentencepiece"
!pip install -q "datasets<2.15.0"

In [ ]:
import json
import os
import pickle
import re
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from datasets import Dataset, DatasetDict

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Configuration

You can tweak these if you want, but the defaults work well.

In [ ]:
# --- Model config ---
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 64
BATCH_SIZE = 64
EPOCHS = 5
LEARNING_RATE = 3e-5
WARMUP_RATIO = 0.1

# --- Categories (must match FinBrain schema) ---
CATEGORIES = [
    "Food & Drink", "Shopping", "Transport", "Bills & Utilities",
    "Entertainment", "Healthcare", "Education", "Housing", "Income", "Other",
]
CATEGORY_MAP = {c: i for i, c in enumerate(CATEGORIES)}
ID2LABEL = {i: c for i, c in enumerate(CATEGORIES)}

# --- FinBrain category map (DoDataThings to ours) ---
DATASET_CATEGORY_MAP = {
    "Restaurants": "Food & Drink",
    "Groceries": "Food & Drink",
    "Shopping": "Shopping",
    "Transportation": "Transport",
    "Utilities": "Bills & Utilities",
    "Insurance": "Bills & Utilities",
    "Entertainment": "Entertainment",
    "Healthcare": "Healthcare",
    "Education": "Education",
    "Rent": "Housing",
    "Mortgage": "Housing",
    "Income": "Income",
    "Travel": "Transport",
    "Subscription": "Entertainment",
    "Personal Care": "Other",
    "Fees": "Other",
    "Transfer": "Other",
}

## Load Training Data

Downloads the DoDataThings 68k realistic US bank transaction dataset.

In [ ]:
print("Downloading DoDataThings dataset...")
df = pd.read_csv(
    "https://huggingface.co/datasets/DoDataThings/us-bank-transaction-categories-v2/"
    "resolve/main/transactions-synthetic.csv"
)
print(f"Loaded {len(df):,} transactions")
print(f"Original categories: {sorted(df['category'].unique())}")

df['our_category'] = df['category'].map(DATASET_CATEGORY_MAP)
unmapped = df['our_category'].isna().sum()
if unmapped:
    print(f"WARNING: {unmapped} transactions with unmapped categories dropped")
    df = df.dropna(subset=['our_category'])

print(f"Mapped to {len(df):,} transactions")
print(f"Our categories: {sorted(df['our_category'].unique())}")
print()
print("Category distribution:")
print(df['our_category'].value_counts())

In [ ]:
texts = df['description'].tolist()
labels = [CATEGORY_MAP[c] for c in df['our_category']]

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.1, random_state=42, stratify=labels
)
print(f"Train: {len(train_texts):,}, Val: {len(val_texts):,}")

train_data = Dataset.from_dict({"text": train_texts, "label": train_labels})
val_data = Dataset.from_dict({"text": val_texts, "label": val_labels})
dataset = DatasetDict({"train": train_data, "validation": val_data})

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

tokenized = dataset.map(tokenize_fn, batched=True)
tokenized = tokenized.remove_columns(["text"])
tokenized.set_format("torch")
print(f"Train: {len(tokenized['train']):,}, Val: {len(tokenized['validation']):,}")

## Fine-tune DistilBERT

Full fine-tuning of DistilBERT takes ~15 min on a T4. The model is small enough (67M params) that LoRA is unnecessary.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(CATEGORIES),
    id2label=ID2LABEL,
    label2id=CATEGORY_MAP,
)

print(f"Model loaded: {model.__class__.__name__}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

training_args = TrainingArguments(
    output_dir="./distilbert-results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none",
    fp16=True,
    dataloader_num_workers=0,
)


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Starting training...")
trainer.train()

In [ ]:
eval_results = trainer.evaluate()
print(f"\nFinal validation accuracy: {eval_results['eval_accuracy']:.4f}")

predictions = trainer.predict(tokenized["validation"])
pred_labels = np.argmax(predictions.predictions, axis=1)
print("\nClassification report:")
print(classification_report(
    tokenized["validation"]["label"],
    pred_labels,
    target_names=CATEGORIES,
))

## Export the Model

Downloads the model as a zip file. You'll extract this into your FinBrain project.

In [ ]:
export_dir = "distilbert-transaction-categorizer"
model.save_pretrained(export_dir)
tokenizer.save_pretrained(export_dir)

wrapper = {
    "model_dir": export_dir,
    "categories": CATEGORIES,
    "max_length": MAX_LENGTH,
}
with open(f"{export_dir}/wrapper_config.json", "w") as f:
    json.dump(wrapper, f, indent=2)

zip_path = "distilbert-transaction-categorizer.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(export_dir):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, export_dir)
            zf.write(file_path, arcname)

from google.colab import files
print(f"Downloading {zip_path}...")
files.download(zip_path)
print(f"\nZip size: {os.path.getsize(zip_path) / 1024 / 1024:.1f} MB")
print("\nDone! Extract to apps/ml-service/data/distilbert-model/")

## Compare: DistilBERT vs Sklearn

This cell tests both models on the validation set and shows where each one wins.

In [ ]:
test_cases = [
    "[debit] STARBUCKS 123 MAIN ST ANYTOWN CA USA",
    "[debit] PYPL*365 MARKET",
    "[debit] AMAZON.COM*K8R2M5VN7",
    "[debit] UBER *TRIP HELP.UBER.COM",
    "SALARY PAYMENT PPD ID: 9901629141",
    "[debit] NETFLIX.COM 123456789",
    "[debit] CVS PHARMACY #8765 ANYTOWN CA",
    "[debit] VERIZON WIRELESS AUTO PAY",
]

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

print(f"{'Test Case':55s} {'DistilBERT':22s} {'Conf':>6s}")
print("-" * 85)

for case in test_cases:
    inputs = tokenizer(case, truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    pred_idx = torch.argmax(probs, dim=-1).item()
    conf = probs[0][pred_idx].item()
    pred_cat = ID2LABEL[pred_idx]
    print(f"{case:55s} {pred_cat:22s} {conf:.3f}")

## Next Steps

1. Download the `distilbert-transaction-categorizer.zip` file (auto-downloads above)
2. On your local machine: `cd apps/ml-service && mkdir -p data/distilbert-model && unzip ~/Downloads/distilbert-transaction-categorizer.zip -d data/distilbert-model/`
3. The sklearn model remains the primary (fast CPU inference)
4. The DistilBERT model serves as a higher-accuracy fallback

---

**Benchmark expected:** DistilBERT should match or slightly exceed the sklearn 99.5% accuracy,
with better generalization to truly unseen merchants (not in the 68k training set).